In [1]:
#load the libraries 
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

In [2]:
#load the data 
df = pd.read_csv('../data/car_fuel_efficiency_2026.csv')
df.head()


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [3]:
#missing values
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [4]:
for col in df.columns:
    print(col)
    print(df[col].unique()[:5])
    print(df[col].nunique())
    print()

model_year
[2006 2008 1996 1989 1994]
50

origin
['Europe' 'Asia' 'USA']
3

fuel_type
['Gasoline' 'Diesel' 'Hybrid']
3

drivetrain
['Front-wheel drive' 'Rear-wheel drive' 'All-wheel drive']
3

num_doors
[4 5 3 2]
4

engine_displacement
[2180 2390 2320 2130 2580]
127

num_cylinders
[6 7 5 8 4]
5

horsepower
[243. 272. 267. 258. 304.]
153

vehicle_weight
[3870 4210 4240 4490 4510]
176

acceleration
[ nan 17.3 18.7 17.5 18.8]
93

fuel_efficiency_mpg
[31.9 31.3 27.5 28.5 31. ]
188



In [5]:
# meadian of horsepower
df['horsepower'].median()

254.0

In [6]:
n = len(df)
n_val = int(n * 0.2)                                                                                                             
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [10]:

cols = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
target = 'fuel_efficiency_mpg'

def split_data(df, seed):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    idx = np.arange(n)
    np.random.seed(seed)
    np.random.shuffle(idx)

    df_shuffled = df.iloc[idx]
    df_train = df_shuffled.iloc[:n_train].reset_index(drop=True)
    df_val = df_shuffled.iloc[n_train:n_train + n_val].reset_index(drop=True)
    df_test = df_shuffled.iloc[n_train + n_val:].reset_index(drop=True)
    return df_train, df_val, df_test

def prepare_X(df, fill_value):
    return df[cols].fillna(fill_value).values

def train_linear_regression(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])   # regularization
    w = np.linalg.inv(XTX).dot(X.T).dot(y)
    return w[0], w[1:]

def predict(X, w0, w):
    return w0 + X.dot(w)

def rmse(y, y_pred):
    return np.sqrt(((y - y_pred) ** 2).mean())

In [11]:
df_train, df_val, df_test = split_data(df, seed=42)
y_train = df_train[target].values
y_val = df_val[target].values

# Option 1: fill with 0
X_train = prepare_X(df_train, 0)
X_val = prepare_X(df_val, 0)
w0, w = train_linear_regression(X_train, y_train)
score_zero = round(rmse(y_val, predict(X_val, w0, w)), 3)

# Option 2: fill with the training mean of horsepower
hp_mean = df_train['horsepower'].mean()
X_train = df_train[cols].fillna({'horsepower': hp_mean}).values
X_val = df_val[cols].fillna({'horsepower': hp_mean}).values
w0, w = train_linear_regression(X_train, y_train)
score_mean = round(rmse(y_val, predict(X_val, w0, w)), 3)

print(score_zero, score_mean)

2.205 2.202


In [12]:
X_train = prepare_X(df_train, 0)
X_val = prepare_X(df_val, 0)

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression(X_train, y_train, r=r)
    print(r, round(rmse(y_val, predict(X_val, w0, w)), 4))

0 2.2053
0.01 2.2058
0.1 2.2241
1 2.3492
5 2.4094
10 2.4195
100 2.4292


In [13]:
scores = []
for seed in range(10):
    df_train, df_val, df_test = split_data(df, seed)
    X_train = prepare_X(df_train, 0)
    X_val = prepare_X(df_val, 0)
    w0, w = train_linear_regression(X_train, df_train[target].values)
    scores.append(rmse(df_val[target].values, predict(X_val, w0, w)))

print(round(np.std(scores), 3))

0.029


In [14]:
df_train, df_val, df_test = split_data(df, seed=9)
df_full = pd.concat([df_train, df_val]).reset_index(drop=True)

X_full = prepare_X(df_full, 0)
y_full = df_full[target].values
X_test = prepare_X(df_test, 0)
y_test = df_test[target].values

w0, w = train_linear_regression(X_full, y_full, r=0.001)
print(round(rmse(y_test, predict(X_test, w0, w)), 3))

2.236
